## Init and load model

In [89]:
import os
%run ../init_project.py
import webbrowser
from pathlib import Path
from torchvista import trace_model
from transformers import GPT2Model, GPT2Tokenizer
import plotly.express as px

gpt2 = GPT2Model.from_pretrained("openai-community/gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token

## Token IDs

### Tokenizing

In [90]:
SAMPLE_INPUTS = [
    "Hello, world",
    "Goodbye, world",
    "How are you?",
] * 10

inputs = tokenizer(
    SAMPLE_INPUTS,
    return_tensors = "pt",
    padding = True,
    truncation = True)

display(inputs)
display(f"B,S: {tuple( inputs['input_ids'].shape )}")

{'input_ids': tensor([[15496,    11,   995, 50256],
        [10248, 16390,    11,   995],
        [ 2437,   389,   345,    30],
        [15496,    11,   995, 50256],
        [10248, 16390,    11,   995],
        [ 2437,   389,   345,    30],
        [15496,    11,   995, 50256],
        [10248, 16390,    11,   995],
        [ 2437,   389,   345,    30],
        [15496,    11,   995, 50256],
        [10248, 16390,    11,   995],
        [ 2437,   389,   345,    30],
        [15496,    11,   995, 50256],
        [10248, 16390,    11,   995],
        [ 2437,   389,   345,    30],
        [15496,    11,   995, 50256],
        [10248, 16390,    11,   995],
        [ 2437,   389,   345,    30],
        [15496,    11,   995, 50256],
        [10248, 16390,    11,   995],
        [ 2437,   389,   345,    30],
        [15496,    11,   995, 50256],
        [10248, 16390,    11,   995],
        [ 2437,   389,   345,    30],
        [15496,    11,   995, 50256],
        [10248, 16390,    11,   995]

'B,S: (30, 4)'

### What does each token ID represent?

In [91]:
hello_world = SAMPLE_INPUTS[0]
print(hello_world)
tokenized = tokenizer.encode(hello_world)
display(tokenized)

for token_id in tokenized:
    print(f"token \"{token_id}\" is: \"{tokenizer.decode(token_id)}\"")

Hello, world


[15496, 11, 995]

token "15496" is: "Hello"
token "11" is: ","
token "995" is: " world"


## Token Embeddings

### Embedding Layer

In [92]:
embedding_layer = gpt2.get_input_embeddings()
display(embedding_layer)
print(f"vocab size: { gpt2.config.vocab_size }")
print(f"embedding size: { gpt2.config.n_embd }")

Embedding(50257, 768)

vocab size: 50257
embedding size: 768


### Embedding Output

In [93]:
embedding_output = embedding_layer(inputs["input_ids"])
print(f"B,S,W: { tuple( embedding_output.shape   )}")

B,S,W: (30, 4, 768)


## Positional Embedding

### Positional Embedding Layer

In [94]:
pos_embedding_layer = gpt2.wpe
display(pos_embedding_layer)
print(f"max seq length: { gpt2.config.n_positions }")
print(f"embedding size: { gpt2.config.n_embd }")

Embedding(1024, 768)

max seq length: 1024
embedding size: 768


### Positional Embedding Output

In [98]:
pos_embedding_output = pos_embedding_layer(torch.arange(inputs["input_ids"].size(1)).unsqueeze(0))
print(f"S,W: {tuple(pos_embedding_output.shape)}")

S,W: (1, 4, 768)


### Visualization

In [100]:
# visualize the positional embedding outputs using a demo input sequence
positions = torch.arange(1024)
display(f"positions shape: {tuple(positions.shape)}")
peo = pos_embedding_layer(positions)
print(f"S,W: {tuple(peo.shape)}")

px.imshow(peo.detach().numpy(), aspect="auto", color_continuous_scale="Viridis")

'positions shape: (1024,)'

S,W: (1024, 768)


In [101]:
# construct transformer block input embeddings (ignore dropout)
embedding_input = embedding_output + pos_embedding_output
print(f"B,S,W: {tuple(embedding_input.shape)}")

B,S,W: (30, 4, 768)


## Transformer Block

### A High Level Overview

In [102]:
block = gpt2.h[0]
trace_model(
    block,
    inputs = embedding_input,
    show_compressed_view=True,
    collapse_modules_after_depth=0,
    height=600,
)

### Layer Norm

In [108]:
layer_norm = block.ln_1
layer_norm.weight

Parameter containing:
tensor([0.2232, 0.1820, 0.1534, 0.1917, 0.2036, 0.1948, 0.1467, 0.1865, 0.2143,
        0.1956, 0.2118, 0.2153, 0.1882, 0.2074, 0.1871, 0.2040, 0.2044, 0.1900,
        0.1952, 0.0475, 0.1909, 0.2115, 0.1971, 0.2202, 0.1998, 0.2108, 0.2303,
        0.1879, 0.1939, 0.2018, 0.1891, 0.1861, 0.1958, 0.1832, 0.1978, 0.2243,
        0.0706, 0.1958, 0.1943, 0.1939, 0.1978, 0.1951, 0.1995, 0.1912, 0.2083,
        0.2037, 0.1849, 0.1945, 0.2189, 0.0419, 0.1977, 0.1979, 0.0608, 0.1824,
        0.2055, 0.0476, 0.1892, 0.2079, 0.2047, 0.2233, 0.2097, 0.2075, 0.2076,
        0.1793, 0.1312, 0.1841, 0.1939, 0.1561, 0.0577, 0.1948, 0.2048, 0.1717,
        0.1942, 0.1708, 0.1989, 0.1993, 0.2082, 0.1071, 0.1968, 0.1770, 0.2164,
        0.1864, 0.1938, 0.2184, 0.1343, 0.1707, 0.0683, 0.1401, 0.1823, 0.2045,
        0.2007, 0.1853, 0.1783, 0.1889, 0.1870, 0.1975, 0.2114, 0.2108, 0.2083,
        0.2409, 0.1938, 0.2022, 0.0857, 0.1823, 0.1879, 0.1979, 0.1850, 0.1029,
        0.1762, 0.

In [111]:
target = (Path.cwd() / "model_visualization.html").absolute()
trace_model(
    gpt2,
    inputs = inputs['input_ids'],
    show_compressed_view=True,
    export_format ="html",
    export_path = target.as_posix(),
    height=2160,
    width=3840,
)
webbrowser.open(target.as_uri(), new=1)

True